## Tokenizer

In [56]:
import re
import logging
import sys

logging.basicConfig(level=logging.INFO, 
                    format="[%(asctime)s] %(levelname)s [%(name)s.%(funcName)s:%(lineno)d] %(message)s",
                    datefmt="%d/%b/%Y %H:%M:%S",
                    stream=sys.stdout)

def flatten(code):
    code = code.replace('\n', ' ')
    code = re.sub(r'\s+', ' ', code)
    return code.strip()

def remove_comments(code):
    code = re.sub(r'\{.*?\}', '', code, flags=re.DOTALL)
    code = flatten(code)
    return code.strip()

def reformat(code):
    SECT_PARAMS = "PARAMETERS" # Parameter section
    SECT_PARAMS_START, SECT_PARAMS_END = "(", ")"

    SECT_VARS = "VARIABLES" # Variable declarations
    SECT_VARS_START, SECT_VARS_END = "[", "]"

    SECT_SUBMODULE = "SUBMODULE" # Submodule starts
    SECT_SUBMODULE_START, SECT_SUBMODULE_END = "<", ">"

    SECT_STATE = "STATE" # State section
    SECT_STATE_START, SECT_STATE_END = "[", "]"
    
    SECT_UNKNOWN = "UNKNOWN" # Unknown section
    
    code = flatten(code)
    code = remove_comments(code)
    
    # Parse sections
    sections = []
    section, buffer, level = None, "", 0
    for index, char in enumerate(code):
        buffer += char
        if section is None:
            level = 0
            if char == SECT_PARAMS_START: 
                logging.info(f"SECT_PARAMS_START {index}")
                section = SECT_PARAMS
            elif char == SECT_VARS_START: 
                logging.info(f"SECT_VARS_START {index}")
                section = SECT_VARS
            elif char == SECT_SUBMODULE_START: 
                logging.info(f"SECT_SUBMODULE_START {index}")
                section = SECT_SUBMODULE
            elif char.isalpha(): 
                logging.info(f"SECT_STATE_START {index}")
                section = SECT_STATE
                level = -1
                
        elif section == SECT_PARAMS:
            if char == SECT_PARAMS_END and level == 0:
                logging.info(f"SECT_PARAMS_END {index}")
                sections.append({
                    "section": SECT_PARAMS,
                    "code": buffer
                })
                buffer, section = "", None
            elif char == SECT_PARAMS_START: level += 1
            elif char == SECT_PARAMS_END: level -= 1
            
        elif section == SECT_VARS:
            if char == SECT_VARS_END and level == 0:
                logging.info(f"SECT_VARS_END {index}")
                sections.append({
                    "section": SECT_VARS,
                    "code": buffer.strip()
                })
                buffer, section = "", None
            elif char == SECT_VARS_START: level += 1
            elif char == SECT_VARS_END: level -= 1
            
        elif section == SECT_SUBMODULE:
            if char == SECT_SUBMODULE_END and level == 0:
                logging.info(f"SECT_SUBMODULE_END {index}")
                sections.append({
                    "section": SECT_SUBMODULE,
                    "code": buffer.strip()
                })
                buffer, section = "", None
            elif char == SECT_VARS_START or char == SECT_PARAMS_START: level += 1
            elif char == SECT_VARS_END or char == SECT_PARAMS_END: level -= 1
        
        else: # SECT_STATE
            if level == -1 and char == SECT_STATE_START:
                sections.append({
                    "section": SECT_STATE,
                    "header": buffer[:-1].strip(),
                    "code": ""
                })
                buffer, level = SECT_STATE_START, 0
            elif level >= 0:
                if char == SECT_STATE_END and level == 0:
                    logging.info(f"SECT_STATE_END {sections[-1]['header']} {index}")
                    sections[-1]["code"] = buffer.strip()
                    buffer, section = "", None
                elif char == SECT_STATE_START: level += 1
                elif char == SECT_STATE_END: level -= 1
    
    # get remaining buffers
    if buffer.strip() != "":
        sections.append({
            "section": SECT_UNKNOWN,
            "code": buffer
        })
    
    # parse subsection
    for section in sections:
        if section["section"] == SECT_VARS:
            _code = section["code"].strip()
            _code = _code.removeprefix(SECT_VARS_START)
            _code = _code.removesuffix(SECT_VARS_END)
            _code = _code.strip()
            _code = re.sub(r';\s+', ';', _code)
            _code = re.sub(r']\s+', ']', _code)
            _code = re.sub(r'\s+\[', '[', _code)
            
            vars, jump_index = [], 0
            for index, char in enumerate(_code):
                if jump_index > index: pass
                else:
                    # class declaration
                    if char == "[":
                        close_index = _code[index:].find("]")
                        # incomplete class declaration
                        if close_index == -1: 
                            jump_index = len(_code)
                        else: jump_index = close_index + index + 1
                        vars.append(_code[index:jump_index].strip())
                    else:
                        close_index = _code[index:].find(";")
                        # unclosed variable declaration line
                        if close_index == -1:
                            jump_index = len(_code)
                        else: jump_index = close_index + index + 1
                        vars.append(_code[index:jump_index].strip())
                        
            section["items"] = vars
            
            
        elif section["section"] == SECT_PARAMS:
            _code = section["code"].strip()
            _code = _code.removeprefix(SECT_PARAMS_START)
            _code = _code.removesuffix(SECT_PARAMS_END)
            
            params = [line.strip() for line in _code.split(";") if line.strip()]
            section["items"] = []
            for param in params:
                space_idx = param.find(" ")
                if space_idx > 1: section["items"].append({
                    "param": param[:space_idx].strip(),
                    "value": param[space_idx:].strip()
                })
                else: section["items"].append({
                    "param": param,
                    "value": None
                })
            
        elif section["section"] == SECT_SUBMODULE:
            _code = section["code"].strip()
            
            _code = _code.removeprefix(SECT_SUBMODULE_START)
            _code = _code.removesuffix(SECT_SUBMODULE_END)
            
            buffer, subcode = "", ""
            for index, char in enumerate(_code):
                buffer += char
                if char == SECT_PARAMS_START or char == SECT_VARS_START:
                    section["header"] = buffer[:-1].strip()
                    subcode = _code[index:].strip()
                    break
            _, section["subsection"] = reformat(subcode)
            
        elif section["section"] == SECT_STATE:
            pass
        
        else: # SECT_UNKNOWN
            pass
        
    return code, sections

import json

# snippet_fn = "DemoDashboard3.SRC"
snippet_fn = "Template.SRC"
with open(f"snippets/{snippet_fn}", "r", encoding="utf-8") as f:
    code = f.read()
    code, section = reformat(code)
    with open(f"cache/{snippet_fn}.json", "w", encoding="utf-8") as ff:
        json.dump(section, ff, ensure_ascii=False, indent=4)
    # print(code)

INFO:root:SECT_PARAMS_START 0
INFO:root:SECT_PARAMS_END 1087
INFO:root:SECT_VARS_START 1089
INFO:root:SECT_VARS_END 2262
INFO:root:SECT_STATE_START 2264
INFO:root:SECT_STATE_END TemplateInit 2323
INFO:root:SECT_STATE_START 2325
INFO:root:SECT_STATE_END Template 3298
INFO:root:SECT_SUBMODULE_START 3300
INFO:root:SECT_SUBMODULE_END 8102
INFO:root:SECT_SUBMODULE_START 8104
INFO:root:SECT_SUBMODULE_END 8486
INFO:root:SECT_SUBMODULE_START 8488
INFO:root:SECT_SUBMODULE_END 8870
INFO:root:SECT_SUBMODULE_START 8872
INFO:root:SECT_SUBMODULE_END 9623
INFO:root:SECT_SUBMODULE_START 9625
INFO:root:SECT_SUBMODULE_END 9964
INFO:root:SECT_SUBMODULE_START 9966
INFO:root:SECT_SUBMODULE_END 11376
INFO:root:SECT_PARAMS_START 0
INFO:root:SECT_PARAMS_END 7
INFO:root:SECT_STATE_START 9
INFO:root:SECT_STATE_END Refresh 4790
INFO:root:SECT_PARAMS_START 0
INFO:root:SECT_PARAMS_END 64
INFO:root:SECT_VARS_START 66
INFO:root:SECT_VARS_END 79
INFO:root:SECT_STATE_START 81
INFO:root:SECT_STATE_END ContributorAdded 